# 03 · Ensemble Tree Methods: Random Forest & XGBoost

Ensemble tree methods are powerful non-parametric regressors that:

* Require **no feature scaling**
* Capture **non-linear interactions** automatically
* Provide **feature importance** diagnostics
* **XGBoost** adds gradient-boosted sequential correction

A key limitation: tree ensembles **cannot extrapolate** beyond the training data range —
they predict the mean of the nearest training leaves. This makes the low-x comparison especially interesting.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams["figure.dpi"] = 120

from data_loader import load_lepton_dis, add_log_features, split_data, get_Xy, low_x_grid
from visualization import plot_coverage, plot_F2_vs_x, comparison_figure

DATA_DIR = "/Users/dikgarg/Desktop/Research/Neutrinos/postdoc/2025/ML/Data/Exp_data/LeptonDIS"

df_raw = load_lepton_dis(DATA_DIR)
df     = add_log_features(df_raw)
df_train, df_test = split_data(df, test_size=0.2, seed=42)

X_train, y_train = get_Xy(df_train)
X_test,  y_test  = get_Xy(df_test)

print(f"Training points: {len(X_train)}  |  Test points: {len(X_test)}")


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb


## Random Forest

In [ ]:
rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    min_samples_leaf=2,
    max_features=1.0,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print("Random Forest")
print(f"  Test MSE : {mean_squared_error(y_test, y_pred_rf):.5f}")
print(f"  Test MAE : {mean_absolute_error(y_test, y_pred_rf):.5f}")
print(f"  Test R2  : {r2_score(y_test, y_pred_rf):.4f}")


## Feature Importance — Random Forest

In [ ]:
feat_names = ["log10(x)", "log10(Q2)"]
importances = rf.feature_importances_

fig, ax = plt.subplots(figsize=(5, 3))
bars = ax.barh(feat_names, importances, color=["#e41a1c", "#377eb8"])
ax.set_xlabel("Mean impurity decrease", fontsize=11)
ax.set_title("Random Forest — feature importance", fontsize=12)
for bar, val in zip(bars, importances):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=10)
plt.tight_layout()
#plt.savefig("../results/figures/03_rf_importance.png", dpi=150)
plt.show()


## XGBoost

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=1.0,
    reg_lambda=1.0,
    random_state=42,
    verbosity=0,
)
xgb_model.fit(X_train, y_train,
              eval_set=[(X_test, y_test)],
              verbose=False)

y_pred_xgb = xgb_model.predict(X_test)
print("XGBoost")
print(f"  Test MSE : {mean_squared_error(y_test, y_pred_xgb):.5f}")
print(f"  Test MAE : {mean_absolute_error(y_test, y_pred_xgb):.5f}")
print(f"  Test R2  : {r2_score(y_test, y_pred_xgb):.4f}")


## Feature Importance — XGBoost

In [ ]:
xgb_imp = xgb_model.feature_importances_

fig, ax = plt.subplots(figsize=(5, 3))
bars = ax.barh(feat_names, xgb_imp, color=["#e41a1c", "#377eb8"])
ax.set_xlabel("Gain", fontsize=11)
ax.set_title("XGBoost — feature importance", fontsize=12)
for bar, val in zip(bars, xgb_imp):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=10)
plt.tight_layout()
#plt.savefig("../results/figures/03_xgb_importance.png", dpi=150)
plt.show()


## Predictions and Low-x Extrapolation

Note: tree methods **plateau** at low x because they cannot extrapolate
beyond the minimum x in the training set — they return the mean of the
most similar training leaf. This is visible as flat tails in the plots below.

In [ ]:
Q2_plot = [1.0, 5.0, 15.0, 30.0]
grids   = low_x_grid(x_min=1e-6, x_max=0.8, n_points=400, Q2_values=Q2_plot)

def build_pred_dict(model, label, color):
    entry = {"label": label, "color": color, "x_arr": None, "Q2_preds": {}}
    for Q2v, (x_arr, X_feat) in grids.items():
        entry["x_arr"]         = x_arr
        entry["Q2_preds"][Q2v] = model.predict(X_feat)
    return entry

preds = [
    build_pred_dict(rf,        "Random Forest", "#e41a1c"),
    build_pred_dict(xgb_model, "XGBoost",       "#377eb8"),
]

fig, _ = comparison_figure(Q2_plot, df, preds)
fig.suptitle("Tree ensemble predictions vs data (note flat low-x tails)", y=1.01)
#plt.savefig("../results/figures/03_tree_predictions.png", dpi=150, bbox_inches="tight")
plt.show()
